# 🏥 MedTrack_DV

# Milestone 2 – Module 3

# Hospital KPI Engineering

## Objective

The objective of this module is to engineer healthcare Key Performance Indicators (KPIs) from the cleaned hospital dataset.

These KPIs will be used in Tableau dashboards for hospital performance monitoring.

## Input

hospital_cleaned.xlsx

## Output

hospital_final_dataset.xlsx

generate_hospital_kpis.py

In [60]:
import pandas as pd
import numpy as np

In [61]:
hospital = pd.read_excel("../data/hospital_cleaned.xlsx")

In [62]:
hospital.shape

(15756, 72)

# KPI 1

## Total Admissions

Total number of patient admissions.

In [63]:
total_admissions = len(hospital)

print(total_admissions)

15756


# KPI 2

## Average Length of Stay

Average duration of hospital stay.

In [64]:
hospital["Avg_Length_of_Stay"] = hospital["DURATION OF STAY"].mean()

# KPI 3

## Occupancy Rate

Percentage of occupied beds within each department.

In [65]:
hospital["Occupancy_Rate"] = (
    hospital["Beds_Occupied"] /
    hospital["Beds_Available"]
) * 100


# KPI 4

## Bed Utilization Rate

Percentage of occupied beds relative to total hospital beds.

In [66]:
hospital["Bed_Utilization"] = (
    hospital["Beds_Occupied"] /
    hospital["Total_Beds"]
) * 100

# KPI 5

## Hospital Admissions

Total admissions for each hospital.

In [67]:
hospital["Hospital_Admissions"] = (
    hospital.groupby(
        "Hospital_ID"
    )["MRD No."]
    .transform("count")
)


# KPI 6

## Department Admissions

Total admissions handled by each department.

In [68]:
hospital["Department_Admissions"] = (
    hospital.groupby(
        ["Hospital_ID","Department_ID"]
    )["MRD No."]
    .transform("count")
)

# KPI 7

## Patients per Nurse

Average patient workload handled by each nurse.

In [69]:
hospital["Patient_per_Nurse"] = (
    hospital["Department_Admissions"] /
    hospital["Nurses"]
).round(2)

# KPI 8

## Patients per Staff

Average patient workload handled by hospital staff.

In [70]:
hospital["Patient_per_Staff"] = (
    hospital["Department_Admissions"] /
    hospital["Staff_Count"]
).round(2)

# KPI 9

## ICU Utilization

Percentage of ICU beds relative to total beds.

In [71]:
hospital["ICU_Utilization"] = (
    hospital["ICU_Beds"] /
    hospital["Total_Beds"]
) * 100

# KPI 10

## Average Length of Stay by Hospital

In [72]:
hospital["Hospital_Avg_Stay"] = hospital.groupby(
    "Hospital_ID"
)["DURATION OF STAY"].transform("mean")

# KPI 11

## Average Length of Stay by Department

In [73]:
hospital["Department_Avg_Stay"] = hospital.groupby(
    ["Hospital_ID","Department_ID"]
)["DURATION OF STAY"].transform("mean")

# KPI 12 - Readmission Analysis

## Objective

The source dataset does not contain a predefined **Readmission** column. Therefore, the Readmission KPI is derived using repeated **MRD No.** values.

### Methodology

- Each **MRD No.** represents a unique patient.
- If the same MRD No. appears more than once, it indicates that the patient was admitted again.
- The first admission is considered the initial admission.
- Every subsequent admission is marked as a readmission.
- The Readmission Rate is calculated as:

Readmission Rate = (Number of Readmission Records / Total Admissions) × 100

This approach is based on the available dataset and provides a data-driven estimation of hospital readmissions.

In [74]:
hospital["MRD No."].duplicated().sum()

np.int64(3513)

In [75]:
hospital[hospital["MRD No."].duplicated(keep=False)] \
    .sort_values(["MRD No.", "D.O.A"]) \
    [["MRD No.", "D.O.A", "D.O.D"]].head(20)

,MRD No.,D.O.A,D.O.D
92,798,2017-05-04,2017-06-04
3525,798,2017-05-10,2017-06-10
15109,798,2019-03-04,2019-03-09
11475,989,2018-10-15,2018-10-17
12718,989,2018-12-05,2018-12-07
12719,989,2018-12-05,2018-12-07
12592,4650,2018-12-01,2018-12-08
12666,4650,2018-12-04,2018-12-06
11529,4973,2018-10-17,2018-10-22
11533,4973,2018-10-17,2018-10-22


In [76]:
hospital = hospital.sort_values(
    ["MRD No.", "D.O.A"]
)

In [77]:
hospital["Readmission"] = hospital.duplicated(
    subset="MRD No.",
    keep="first"
).astype(int)

In [78]:
hospital["Readmission"].value_counts()

Readmission
0    12243
1     3513
Name: count, dtype: int64

In [79]:
readmission_rate = (
    hospital["Readmission"].sum()
    /
    len(hospital)
) * 100

print(f"Readmission Rate: {readmission_rate:.2f}%")

Readmission Rate: 22.30%


In [80]:
hospital["Readmission_Rate"] = readmission_rate

# View Engineered Dataset

In [81]:
hospital.head()

,SNO,MRD No.,D.O.A,D.O.D,AGE,GENDER,RURAL,TYPE OF ADMISSION-EMERGENCY/OPD,month year,DURATION OF STAY,...,Bed_Utilization,Hospital_Admissions,Department_Admissions,Patient_per_Nurse,Patient_per_Staff,ICU_Utilization,Hospital_Avg_Stay,Department_Avg_Stay,Readmission,Readmission_Rate
15194,15196,506,2019-03-07,2019-03-10,50,M,U,O,2019-03-01,4,...,11.666667,1064,882,32.67,14.46,15.000000,6.172932,6.164399,0,22.296268
92,93,798,2017-05-04,2017-06-04,71,M,U,O,2017-04-01,2,...,11.666667,974,806,32.24,13.00,13.333333,6.637577,6.663772,0,22.296268
3525,3526,798,2017-05-10,2017-06-10,71,M,U,O,2017-10-01,2,...,7.444444,1370,1142,43.92,17.84,16.666667,6.466423,6.543783,1,22.296268
15109,15111,798,2019-03-04,2019-03-09,72,M,U,O,2019-03-01,6,...,12.545455,990,822,29.36,13.26,15.454545,6.467677,6.540146,1,22.296268
11475,11477,989,2018-10-15,2018-10-17,71,M,U,E,2018-10-01,3,...,8.375000,1253,1011,36.11,16.05,15.000000,6.465283,6.503462,0,22.296268


In [82]:
hospital.columns

Index(['SNO', 'MRD No.', 'D.O.A', 'D.O.D', 'AGE', 'GENDER', 'RURAL',
       'TYPE OF ADMISSION-EMERGENCY/OPD', 'month year', 'DURATION OF STAY',
       'duration of intensive unit stay', 'OUTCOME', 'SMOKING ', 'ALCOHOL',
       'DM', 'HTN', 'CAD', 'PRIOR CMP', 'CKD', 'HB', 'TLC', 'PLATELETS',
       'GLUCOSE', 'UREA', 'CREATININE', 'BNP', 'RAISED CARDIAC ENZYMES', 'EF',
       'SEVERE ANAEMIA', 'ANAEMIA', 'STABLE ANGINA', 'ACS', 'STEMI',
       'ATYPICAL CHEST PAIN', 'HEART FAILURE', 'HFREF', 'HFNEF', 'VALVULAR',
       'CHB', 'SSS', 'AKI', 'CVA INFRACT', 'CVA BLEED', 'AF', 'VT', 'PSVT',
       'CONGENITAL', 'UTI', 'NEURO CARDIOGENIC SYNCOPE', 'ORTHOSTATIC',
       'INFECTIVE ENDOCARDITIS', 'DVT', 'CARDIOGENIC SHOCK', 'SHOCK',
       'PULMONARY EMBOLISM', 'CHEST INFECTION', 'Hospital_ID', 'Department',
       'Department_ID', 'Hospital_Name', 'Hospital_Type', 'City', 'State',
       'Total_Beds', 'ICU_Beds', 'Doctor_ID', 'Doctor_Name', 'Nurses',
       'Staff_Count', 'Beds_Available', 

# Validate Missing Values

In [83]:
hospital.isnull().sum()

SNO                    0
MRD No.                0
D.O.A                  0
D.O.D                  0
AGE                    0
                      ..
ICU_Utilization        0
Hospital_Avg_Stay      0
Department_Avg_Stay    0
Readmission            0
Readmission_Rate       0
Length: 84, dtype: int64

# Export Final Dataset

Save the KPI-engineered dataset for Tableau dashboard development.

In [84]:
hospital.to_excel(
    "../data/hospital_final_dataset.xlsx",
    index=False
)

# Module 3 Summary

## KPIs Generated

- Total Admissions
- Average Length of Stay
- Occupancy Rate
- Bed Utilization Rate
- Hospital Admissions
- Department Admissions
- Patient per Nurse
- Patient per Staff
- ICU Utilization
- Hospital Average Stay
- Department Average Stay
- Readmission Rate

## Deliverables

- hospital_final_dataset.xlsx
- generate_hospital_kpis.py

The engineered dataset is now ready for Module 4 (Dashboard Planning) and Tableau dashboard development.